# Subagent 도구 위임

중앙 에이전트가 전문 에이전트에 필요한 맥락만 전달하면 호출 경계가 어떻게 드러날까요?


## 실행 환경 준비

Python 3.12와 고정된 패키지 환경을 사용합니다. 키 값은 출력하지 않고 존재 여부만 확인합니다.


In [1]:
# 고정한 패키지를 설치하고 비밀 키의 존재 여부만 확인합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3
from pathlib import Path
import os
from importlib.metadata import version
from dotenv import load_dotenv

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break

MODEL = "gpt-5.4-mini"
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
print({"langchain": version("langchain"), "langgraph": version("langgraph"), "model": MODEL, "cost_limit_usd": 1.0})


Note: you may need to restart the kernel to use updated packages.
{'langchain': '1.4.0', 'langgraph': '1.2.11', 'model': 'gpt-5.4-mini', 'cost_limit_usd': 1.0}


## 전문 에이전트를 도구로 감싸기


In [2]:
# 전문 에이전트를 도구로 감싸고 중앙 supervisor에 등록합니다.
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool

model = init_chat_model(f"openai:{MODEL}", reasoning_effort="none")
calls: list[str] = []
specialist = create_agent(model, tools=[], system_prompt="환불 정책 확인 항목을 두 문장으로 답하세요.")

@tool
def policy_specialist(question: str) -> str:
    """환불 정책 질문을 검토하고 핵심 근거만 반환합니다."""
    calls.append("policy_specialist")
    result = specialist.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].text

supervisor = create_agent(model, tools=[policy_specialist], system_prompt="환불이 포함된 요청에는 policy_specialist 도구를 반드시 한 번 사용한 뒤 고객 답변을 작성하세요.")


## 중앙 에이전트가 결과 종합


In [3]:
# supervisor를 실제 실행하고 메시지 기록에서 도구 호출을 확인합니다.
request = "배송 지연 주문의 환불 가능 여부와 고객 안내 문구를 작성해 주세요."
result = supervisor.invoke({"messages": [{"role": "user", "content": request}]})
tool_names = [call["name"] for message in result["messages"] for call in getattr(message, "tool_calls", [])]
print("모델이 선택한 도구:", tool_names)
print("도구 실행 기록:", calls)
print("최종 답변:", result["messages"][-1].text)
assert tool_names == ["policy_specialist"] and calls == ["policy_specialist"]


모델이 선택한 도구: ['policy_specialist']
도구 실행 기록: ['policy_specialist']
최종 답변: 네, 아래처럼 안내하시면 됩니다.

### 1) 배송 지연 주문 환불 가능 여부
- **가능한 경우**: 약정된 출고/배송 기한을 **현저히 초과**했고, 고객이 **배송 지연을 사유로 취소 및 환불을 요청**한 경우
- **어려운 경우**:  
  - 고객 **주소 오류**
  - **수취 거부**
  - **연락 두절**
  - 이미 **배송 완료**된 경우
  - 배송이 **정상적으로 진행 중**인 경우

### 2) 고객 안내 문구 예시
> 안녕하세요, 고객님.  
> 현재 주문 상품은 배송이 지연되고 있습니다. 지연 사유는 [사유 입력]이며, 현재 예상 배송일은 [예정일 입력]입니다.  
> 약정된 출고/배송 기한을 현저히 초과한 경우 또는 미배송 상태에서 고객님께서 주문 취소를 요청하시면 환불이 가능합니다.  
> 다만 주소 오류, 수취 거부, 연락 두절 등 고객 귀책 사유가 있거나 이미 배송이 완료된 경우에는 환불이 어려울 수 있습니다.  
> 불편을 드려 죄송하며, 추가 확인이 필요하시면 안내드리겠습니다.

원하시면 제가 이 문구를 **더 친절한 톤 / 더 단호한 톤 / CS 매뉴얼용 문구**로도 바꿔드릴게요.


## 전달 맥락 줄이기


In [4]:
# 전체 기록과 전문 에이전트에 전달한 질문의 문자 수를 비교합니다.
full_context = request + "\n" + "내부 기록 " * 100
focused_context = request
print({"전체_문자수": len(full_context), "전달_문자수": len(focused_context)})
assert len(focused_context) < len(full_context)


{'전체_문자수': 639, '전달_문자수': 38}
